# 05c. 두 번째 캐스터 다시 뽑기 — **Colab GPU**

caster_m20_v2 탈락 이유: 문장 끝을 억지로 올리고, 억지로 활기찬 느낌 (v1은 소리 지르는 느낌)
→ 설명에서 "bright, lively, energetic" 같은 단어를 빼고, **차분한 진행 + 끝음이 자연스럽게 내려감**을 명시한다.
→ Base 복제는 참조 음성의 억양까지 따라 하므로, **참조 문장도 평서문(~입니다)으로 끝나게** 바꾼다. (기존 문장은 "~되는데요"로 끝나서 끝을 올리기 쉬움)

1. 후보 4개 생성 → Drive `voices/caster_candidates/`
2. 참조 음성만 듣지 않고, **Base로 복제 합성한 중계 문장**을 해설위원과 주고받는 형태로 들어보기 (v2는 참조는 괜찮았는데 합성 후에 망가졌음)
3. 고른 후보로 `voices.json`의 caster_m20_v2를 교체

In [ ]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U qwen-tts

In [ ]:
import gc
import json
import shutil
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from google.colab import drive
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

drive.mount("/content/drive")
VOICE_DIR = Path("/content/drive/MyDrive/kbo-listen/voices")
CAND = VOICE_DIR / "caster_candidates"
CAND.mkdir(parents=True, exist_ok=True)
VOICES = json.loads((VOICE_DIR / "voices.json").read_text(encoding="utf-8"))

# 새 참조 문장: 직접 쓴 일반적인 문장, 모두 평서문으로 끝냄 (실제 중계 원문 아님)
NEW_REF_TEXT = "안녕하십니까, 오늘 프로야구 중계를 맡은 캐스터입니다. 양 팀 선발 투수가 마운드에 오릅니다. 1회초 공격부터 시작하겠습니다."


def load(name):
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")


def free():
    gc.collect()
    torch.cuda.empty_cache()


print(sorted(p.name for p in VOICE_DIR.iterdir()))
print(list(VOICES))

In [ ]:
# 1) 캐스터 후보 4개. 이미 만든 후보는 다시 만들지 않는다 (다시 뽑으려면 이름을 바꾸거나 파일을 지움)
# 공통: 에너지·밝음을 강조하는 단어는 쓰지 않고, 평서문 끝음이 내려간다는 것과 보통 음량을 명시
COMMON = ("Normal speaking volume, no forced excitement. Declarative sentences end with a natural "
          "falling intonation, never rising or sing-song at the end.")
CANDIDATES = {  # 이름: (설명, 시드)
    "caster_m38_steady": ("A Korean male baseball play-by-play announcer in his late 30s, mid-low voice, "
                          "steady and matter-of-fact delivery at a moderate pace. " + COMMON, 20),
    "caster_m40_baritone": ("A Korean male baseball play-by-play announcer in his 40s, warm baritone voice, "
                            "brisk but calm delivery, relaxed and confident. " + COMMON, 21),
    "caster_m35_anchor": ("A Korean male sports announcer in his mid 30s, clean and plain voice like a news anchor, "
                          "precise articulation, even and composed tone. " + COMMON, 22),
    "caster_m42_relaxed": ("A Korean male veteran baseball announcer in his early 40s, slightly husky mid-low voice, "
                           "easygoing conversational delivery, unhurried. " + COMMON, 23),
}

design = None
cand_refs = {}
for name, (desc, seed) in CANDIDATES.items():
    path = CAND / f"ref_{name}.wav"
    if path.exists():
        wav, sr = sf.read(path, dtype="float32")
    else:
        if design is None:
            design = load("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
        torch.manual_seed(seed)
        wavs, sr = design.generate_voice_design(text=NEW_REF_TEXT, language="Korean", instruct=desc)
        wav = wavs[0]
        sf.write(path, wav, sr)
    cand_refs[name] = (wav, sr)
    print(name)
    display(Audio(wav, rate=sr))

if design is not None:
    del design
    free()

In [ ]:
# 2) 복제 합성으로 듣기: 실제 데이터처럼 Base 모델로 후보 목소리를 복제해서 중계 문장을 읽힌다
# 비교용으로 확정 캐스터 caster_m30도 같은 대본으로 들려줌
clone = load("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

DIALOGUE = [  # (화자, 문장) C=캐스터, A=해설위원. 억양이 드러나는 평서문 + 흥분하는 장면을 섞음
    ("C", "3회말, 기아 공격입니다. 스코어 기아 1 대 LG 2."),
    ("C", "원아웃, 타석에는 김도영."),
    ("C", "초구, 바깥쪽 빠졌고요."),
    ("A", "네, 유인구에 잘 참았습니다."),
    ("C", "몸쪽 직구, 스트라이크."),
    ("C", "먹혔고 파울입니다."),
    ("C", "볼카운트 원앤투."),
    ("A", "투 스트라이크 이후에는 아무래도 타자가 불리해지죠."),
    ("C", "투수 피치클락 위반, 볼."),
    ("C", "큰 타구! 넘어갑니다! 홈런!"),
    ("A", "그렇죠, 노림수가 정확했어요."),
]
GAP = 0.5


def synth(texts, prompt):
    wavs, sr = clone.generate_voice_clone(text=texts, language=["Korean"] * len(texts), voice_clone_prompt=prompt)
    return [np.asarray(w, dtype=np.float32) for w in wavs], sr


def ref_of(name):
    return sf.read(VOICE_DIR / f"ref_{name}.wav", dtype="float32")


analyst = "analyst_a40_warm"
analyst_prompt = clone.create_voice_clone_prompt(ref_audio=ref_of(analyst), ref_text=VOICES[analyst]["ref_text"])
analyst_wavs, sr = synth([t for s, t in DIALOGUE if s == "A"], analyst_prompt)

casters = {"caster_m30 (비교용)": (ref_of("caster_m30"), VOICES["caster_m30"]["ref_text"])}
casters.update({name: (ref, NEW_REF_TEXT) for name, ref in cand_refs.items()})

for name, (ref, ref_text) in casters.items():
    prompt = clone.create_voice_clone_prompt(ref_audio=ref, ref_text=ref_text)
    caster_wavs, _ = synth([t for s, t in DIALOGUE if s == "C"], prompt)
    c_iter, a_iter = iter(caster_wavs), iter(analyst_wavs)
    pieces = []
    for speaker, _ in DIALOGUE:  # 대사 순서대로 화자별 음성을 꺼내 이어 붙임
        pieces += [next(c_iter if speaker == "C" else a_iter), np.zeros(int(GAP * sr), np.float32)]
    print(name)
    display(Audio(np.concatenate(pieces), rate=sr))

In [ ]:
# 3) 캐스터 교체: 기존 2명(m30, m20_v2)을 빼고 새 후보 3명으로 → 캐스터 3 × 해설위원 2 = 6가지 조합
# m42_relaxed는 탈락. 기존 참조 음성은 지우지 않고 후보 폴더로 옮겨 둠 (파일럿 v3 재현용)
CHOSEN = ["caster_m38_steady", "caster_m40_baritone", "caster_m35_anchor"]
OLD = ["caster_m30", "caster_m20_v2"]

for name in CHOSEN:
    shutil.copy(CAND / f"ref_{name}.wav", VOICE_DIR / f"ref_{name}.wav")
for name in OLD:
    old_path = VOICE_DIR / f"ref_{name}.wav"
    if old_path.exists():
        shutil.move(old_path, CAND / old_path.name)

# 해설위원 항목은 그대로 두고 캐스터 항목만 새로 씀 (여러 번 실행해도 결과가 같음)
VOICES = {name: info for name, info in VOICES.items() if info["role"] != "caster"}
VOICES.update({name: {"role": "caster", "ref_text": NEW_REF_TEXT} for name in CHOSEN})
(VOICE_DIR / "voices.json").write_text(json.dumps(VOICES, ensure_ascii=False, indent=1), encoding="utf-8")
print(sorted(p.name for p in VOICE_DIR.iterdir()))
print(json.dumps(VOICES, ensure_ascii=False, indent=1))